# 04 - RQ2: Is route-month load factor associated with route-month late-arrival rate?

* **IV:** load factor (T-100: passengers / seats x 100) for a carrier-route-month, continuous.
* **DV:** late-arrival rate (share of that route-month's operated flights with `ArrDel15 = 1`), **kept continuous** (not binarised).
* **H0:** rho = 0 (no linear association).  **H1:** rho != 0.
* **Tests:** Pearson r with Fisher-z confidence interval (primary); Spearman rho (robust to outliers);
  simple linear regression (direction and size of the slope); route-clustered and controlled regressions as robustness checks.

*Why clustered standard errors?* The same route appears in up to 12 months, so observations are not fully independent;
the plain Pearson p-value is therefore optimistic. The clustered regression corrects for that.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from config import *
from results_store import save_result, load_results
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from matplotlib.colors import LinearSegmentedColormap
from plot_style import set_style, save_fig
set_style()
blue_seq = LinearSegmentedColormap.from_list("blue_seq", ["#ffffff"] + BLUE_RAMP)

rma = pd.read_csv(PROCESSED / "route_month_analysis.csv")
rma["route"] = rma["carrier_id"].astype(str) + "_" + rma["origin_id"].astype(str) + "_" + rma["dest_id"].astype(str)
x, y, n = rma["load_factor"], rma["late_rate"], len(rma)
print(f"{n:,} route-months across {rma['route'].nunique():,} carrier-routes (minimum required: {load_results()['sample_size']['rq2']})")

148,241 route-months across 9,998 carrier-routes (minimum required: 783)


In [2]:
# --- correlation ---------------------------------------------------------------------------------
r, p_r = stats.pearsonr(x, y)
zf, se = np.arctanh(r), 1 / np.sqrt(n - 3)
r_lo, r_hi = np.tanh(zf - 1.96 * se), np.tanh(zf + 1.96 * se)
rho, p_rho = stats.spearmanr(x, y)
t_stat = r * np.sqrt(n - 2) / np.sqrt(1 - r ** 2)
print(f"Pearson r = {r:.4f} (95% CI {r_lo:.4f} to {r_hi:.4f}), t({n-2}) = {t_stat:.2f}, p = {p_r:.3g}")
print(f"Spearman rho = {rho:.4f}, p = {p_rho:.3g}")

Pearson r = 0.1078 (95% CI 0.1028 to 0.1129), t(148239) = 41.76, p = 0
Spearman rho = 0.0986, p = 0


In [3]:
# --- simple linear regression ----------------------------------------------------------------
X = sm.add_constant(x)
ols = sm.OLS(y, X).fit()
ols_cl = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(rma["route"])[0]})
ci = ols_cl.conf_int()
print(ols.summary().tables[1])
print(f"\nclustered SE for slope: {ols_cl.bse['load_factor']:.4f}  p = {ols_cl.pvalues['load_factor']:.3g}")

# assumption checks
resid = ols.resid
bp_lm, bp_p, _, _ = sm.stats.diagnostic.het_breuschpagan(resid, X)
infl = ols.get_influence(); cooks = infl.cooks_distance[0]
n_infl = int((cooks > 4 / n).sum())
keep = cooks <= 4 / n
ols_noinf = sm.OLS(y[keep], sm.add_constant(x[keep])).fit()
print(f"residual skew {stats.skew(resid):.2f}, excess kurtosis {stats.kurtosis(resid):.2f}; Breusch-Pagan p = {bp_p:.3g}; "
      f"influential points (Cook > 4/n): {n_infl} ({n_infl/n*100:.1f}%); slope without them = {ols_noinf.params['load_factor']:.4f}")

                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          11.8197      0.228     51.828      0.000      11.373      12.267
load_factor     0.1181      0.003     41.759      0.000       0.113       0.124

clustered SE for slope: 0.0058  p = 1.05e-93
residual skew 0.81, excess kurtosis 1.00; Breusch-Pagan p = 3.25e-88; influential points (Cook > 4/n): 6710 (4.5%); slope without them = 0.1205


In [4]:
# --- robustness: weighted and controlled models ---------------------------------------------------
wls = sm.WLS(y, X, weights=rma["flights_operated"]).fit(cov_type="HC3")
rma["log_dist"] = np.log(rma["distance_otp"].clip(lower=1))
form = "late_rate ~ load_factor + log_dist + C(carrier) + C(month)" + (" + C(year)" if rma["year"].nunique() > 1 else "")
ctrl = smf.ols(form, data=rma).fit(
    cov_type="cluster", cov_kwds={"groups": pd.factorize(rma["route"])[0]})
rob = pd.DataFrame([
    ["Simple OLS (route-clustered SE)", ols_cl.params["load_factor"], ols_cl.bse["load_factor"], ols_cl.pvalues["load_factor"], ols.rsquared],
    ["Weighted by flights (HC3 SE)", wls.params["load_factor"], wls.bse["load_factor"], wls.pvalues["load_factor"], wls.rsquared],
    ["Without influential points", ols_noinf.params["load_factor"], ols_noinf.bse["load_factor"], ols_noinf.pvalues["load_factor"], ols_noinf.rsquared],
    ["+ distance, carrier, month, year (clustered SE)", ctrl.params["load_factor"], ctrl.bse["load_factor"], ctrl.pvalues["load_factor"], ctrl.rsquared],
], columns=["Model", "Slope (pts late rate per +1 pt load factor)", "SE", "p-value", "R2"])
rob.to_csv(TAB / "rq2_robustness.csv", index=False)
rob.round(4)

,Model,Slope (pts late rate per +1 pt load factor),SE,p-value,R2
0,Simple OLS (route-clustered SE),0.1181,0.0058,0.0,0.0116
1,Weighted by flights (HC3 SE),0.1622,0.0031,0.0,0.0213
2,Without influential points,0.1205,0.0027,0.0,0.0136
3,"+ distance, carrier, month, year (clustered SE)",0.0761,0.0060,0.0,0.1907


In [5]:
# --- figures ---------------------------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(6.5, 3.2))
hb = ax[0].hexbin(x, y, gridsize=35, cmap=blue_seq, mincnt=1, extent=(x.min(), 100, 0, y.quantile(.995)))
xs = np.linspace(x.min(), x.max(), 50)
pred = ols.get_prediction(sm.add_constant(pd.Series(xs, name="load_factor"))).summary_frame(alpha=0.05)
ax[0].plot(xs, pred["mean"], color=PALETTE[1], lw=2)
ax[0].fill_between(xs, pred["mean_ci_lower"], pred["mean_ci_upper"], color=PALETTE[1], alpha=0.25, lw=0)
ax[0].set_xlabel("Load factor (%)"); ax[0].set_ylabel("Late-arrival rate (%)")
fig.colorbar(hb, ax=ax[0], pad=0.02).set_label("Route-months", fontsize=8)
hb2 = ax[1].hexbin(ols.fittedvalues, resid, gridsize=30, cmap=blue_seq, mincnt=1)
ax[1].axhline(0, color=PALETTE[1], lw=1.4)
ax[1].set_xlabel("Fitted late-arrival rate (%)"); ax[1].set_ylabel("Residual (percentage points)")
fig.tight_layout(); save_fig(fig, "fig16_rq2_regression")

rq2 = {
    "n": int(n), "n_routes": int(rma["route"].nunique()), "min_required": load_results()["sample_size"]["rq2"],
    "pearson_r": float(r), "pearson_ci": [float(r_lo), float(r_hi)], "pearson_t": float(t_stat), "pearson_p": float(p_r), "r_squared": float(r ** 2),
    "spearman_rho": float(rho), "spearman_p": float(p_rho),
    "slope": float(ols.params["load_factor"]), "slope_se_naive": float(ols.bse["load_factor"]),
    "slope_ci_clustered": [float(ci.loc["load_factor", 0]), float(ci.loc["load_factor", 1])],
    "slope_se_clustered": float(ols_cl.bse["load_factor"]), "slope_p_clustered": float(ols_cl.pvalues["load_factor"]),
    "intercept": float(ols.params["const"]),
    "resid_skew": float(stats.skew(resid)), "resid_kurtosis": float(stats.kurtosis(resid)), "bp_p": float(bp_p),
    "influential_n": n_infl, "influential_pct": float(n_infl / n * 100), "slope_without_influential": float(ols_noinf.params["load_factor"]),
    "controlled_slope": float(ctrl.params["load_factor"]), "controlled_p": float(ctrl.pvalues["load_factor"]),
    "weighted_slope": float(wls.params["load_factor"]), "weighted_p": float(wls.pvalues["load_factor"]),
    "late_rate_mean": float(y.mean()), "lf_mean": float(x.mean()),
    "effect_label": "negligible" if abs(r) < 0.1 else "small" if abs(r) < 0.3 else "medium" if abs(r) < 0.5 else "large",
    "decision": "reject H0" if p_r < 0.05 else "fail to reject H0",
}
save_result("rq2", rq2)
print(rq2["decision"], "| r =", round(r, 3), "| slope =", round(rq2["slope"], 4), "pts per +1 pt load factor")

reject H0 | r = 0.108 | slope = 0.1181 pts per +1 pt load factor
